# StyleStudio (CVPR 2025) — chạy trên Kaggle GPU

Text-Driven Style Transfer with Selective Control of Style Elements.

**Bản này dùng fork `namt9/StyleStudio`, nhánh `adaptive-fusion`** — đã có sẵn `FusionController` (dừng fusion thích ứng thay vì `end_fusion` cố định) và đã fix bug `end_fusion` không được forward vào `generate()`. Xem chi tiết: `docs/superpowers/plans/2026-07-13-adaptive-end-fusion.md`.

## ⚙️ TRƯỚC KHI CHẠY — cấu hình notebook (menu bên phải)
1. **Settings → Accelerator → `GPU T4 x2`** (hoặc `GPU P100`). Bắt buộc — code cần CUDA.
2. **Settings → Internet → `On`** (cần tài khoản đã xác thực SĐT) để tải model từ HuggingFace.

Chạy lần lượt từng cell từ trên xuống.

## 1. Kiểm tra GPU

In [1]:
!nvidia-smi
import torch
print('torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chưa bật GPU! Vào Settings -> Accelerator -> GPU T4 x2'

Fri Jul 10 12:01:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone source code

In [ ]:
import os
REPO_DIR = '/kaggle/working/StyleStudio'
if not os.path.exists(REPO_DIR):
    !git clone -b adaptive-fusion https://github.com/namt9/StyleStudio.git {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -6
!ls

## 3. Cài dependencies
Giữ nguyên `torch` có sẵn của Kaggle (đã kèm CUDA), chỉ cài các thư viện được pin trong `requirements.txt`. KHÔNG cài lại torch/torchvision để tránh phá vỡ CUDA.

In [3]:
!pip install -q \
  diffusers==0.25.1 \
  transformers==4.45.2 \
  tokenizers==0.20.1 \
  huggingface-hub==0.24.6 \
  accelerate peft safetensors einops omegaconf opencv-python
print('done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 113.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.0 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.24.6 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.24.6 which is incompatible.
done


## 4. Tải checkpoint cần đường dẫn cục bộ + vá source

- `stabilityai/stable-diffusion-xl-base-1.0` và `madebyollin/sdxl-vae-fp16-fix`: là repo HF hợp lệ → code tự tải khi chạy, không cần làm gì.
- **Image encoder** (`h94/IP-Adapter/.../image_encoder`) và **CSGO ckpt** (`InstantX/CSGO/csgo_4_32.bin`): source dùng chúng như đường dẫn cục bộ → phải tải trước rồi trỏ lại.

Cell này cũng **xóa dòng HF mirror Trung Quốc** (`hf-mirror.com`) vốn sẽ làm tải model bị chậm/lỗi trên Kaggle.

In [ ]:
import os, time
# Tat backend "Xet" cua huggingface_hub - hay bi 403 khi tai an danh (khong token)
# tren cac repo lon nhu SDXL base. Phai dat TRUOC moi lan import/goi huggingface_hub.
os.environ['HF_HUB_DISABLE_XET'] = '1'
!pip uninstall -y hf_xet -q

from huggingface_hub import hf_hub_download, snapshot_download, login
import re

# --- Dang nhap HuggingFace neu co token (khac phuc 403 tren CDN Xet khi tai an danh) ---
# Tao token (role Read) tai https://huggingface.co/settings/tokens, roi dien vao day
# hoac chay truoc: os.environ['HF_TOKEN'] = 'hf_...'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Da dang nhap HuggingFace bang token.')
else:
    print('CANH BAO: chua co HF_TOKEN. Neu gap loi 403 khi tai model, tao token (role Read) '
          'tai https://huggingface.co/settings/tokens, dien vao bien HF_TOKEN o tren, '
          'roi Restart session va chay lai tu cell nay.')

# Bo qua anh preview/docs khong can thiet cho pipeline (giam so file phai tai, giam rui ro 403)
SKIP_ASSETS = ['*.png', '*.jpg', '*.jpeg', '*.gif', '*.md', '*.PNG', '*.JPG']

def snapshot_download_retry(repo_id, retries=5, delay=15, **kw):
    """max_workers=1 (tuan tu, khong tai song song) de tranh CDN rate-limit 403
    khi snapshot_download mac dinh tai nhieu file cung luc. Kem retry vi 403 co
    the la loi tam thoi cua CDN."""
    for attempt in range(1, retries + 1):
        try:
            return snapshot_download(repo_id, max_workers=1, **kw)
        except Exception as e:
            print(f'  [{repo_id}] lan {attempt}/{retries} loi: {e}')
            if attempt == retries:
                raise
            time.sleep(delay)

# --- Tai truoc SDXL base + VAE (single-thread, co retry, bo qua anh/docs) de cache san,
#     tranh 403 do rate-limit khi from_pretrained() tu tai song song sau nay ---
print('Dang tai truoc SDXL base (co the mat vai phut)...')
snapshot_download_retry('stabilityai/stable-diffusion-xl-base-1.0', ignore_patterns=SKIP_ASSETS)
snapshot_download_retry('madebyollin/sdxl-vae-fp16-fix', ignore_patterns=SKIP_ASSETS)
print('Da cache SDXL base + VAE.')

# --- Tải image encoder (chỉ subfolder cần thiết) ---
ie_root = snapshot_download_retry('h94/IP-Adapter', allow_patterns=['sdxl_models/image_encoder/*'])
IMAGE_ENCODER_LOCAL = os.path.join(ie_root, 'sdxl_models', 'image_encoder')
print('image_encoder:', IMAGE_ENCODER_LOCAL)

# --- Tải CSGO checkpoint ---
CSGO_LOCAL = hf_hub_download('InstantX/CSGO', 'csgo_4_32.bin')
print('csgo ckpt   :', CSGO_LOCAL)

# --- Vá các script inference: bỏ HF mirror + trỏ đường dẫn cục bộ ---
def patch(path):
    with open(path) as f:
        s = f.read()
    s = s.replace("os.environ['HF_ENDPOINT']='https://hf-mirror.com'",
                  "# HF mirror removed for Kaggle")
    s = s.replace('image_encoder_path = "h94/IP-Adapter/sdxl_models/image_encoder"',
                  f'image_encoder_path = "{IMAGE_ENCODER_LOCAL}"')
    s = s.replace("csgo_ckpt ='InstantX/CSGO/csgo_4_32.bin'",
                  f"csgo_ckpt ='{CSGO_LOCAL}'")
    with open(path, 'w') as f:
        f.write(s)
    print('patched:', path)

for p in ['infer_StyleStudio.py', 'infer_StyleStudio_layout_stability.py']:
    if os.path.exists(p):
        patch(p)

## 5. Chạy StyleStudio

Bật đủ 3 đóng góp chính của paper:
- `--adainIP`  → Cross-Modal AdaIN
- `--fuSAttn` + `--end_fusion 20` → Teacher Model (ổn định layout, `end_fusion` cố định)
- **hoặc** `--fuSAttn --adaptive_fusion --rho 0.2` → Teacher Model với dừng fusion **thích ứng** (nhánh `adaptive-fusion`), không cần dò tay `end_fusion`.

Lần chạy đầu sẽ tải SDXL (~vài GB) nên **chậm** (5–15 phút). Ảnh kết quả lưu tại `test.jpg`.

> Cell vá OOM bằng SDPA q/k-sharing (bản cũ) đã được **bỏ** — nó không tương thích với adaptive fusion vì `FusionController` cần `attn_probs` được materialize đầy đủ mỗi bước fusion để đo độ lệch teacher/student. OOM mitigation giờ dựa vào `enable_attention_slicing()` (mục xử lý lỗi cuối notebook) + early-stop của adaptive fusion.

In [ ]:
# Patch infer scripts (chay 1 lan, idempotent):
#  1) expose --style_scale / --guidance_scale (chi infer_StyleStudio.py)
#  2) forward end_fusion vao loi goi generate().
#     Nhanh 'adaptive-fusion' da fix bug nay cho infer_StyleStudio.py (patch_forward_end_fusion
#     se in "da forward end_fusion" va khong doi gi - dung nhu ky vong, khong phai loi).
#     infer_StyleStudio_layout_stability.py (dung o muc 8C ben duoi) CHUA duoc fix -> van can patch.
def patch_forward_end_fusion(path, indent):
    s = open(path).read()
    needle = 'num_inference_steps=args.num_inference_steps,'
    block  = needle + '\n' + indent + 'end_fusion=args.end_fusion,'
    if block in s:
        print('  da forward end_fusion ->', path); return
    assert needle in s, f'khong tim thay needle trong {path}'
    open(path,'w').write(s.replace(needle, block, 1))
    print('  forwarded end_fusion ->', path)

# infer_StyleStudio.py: expose style_scale/guidance_scale + forward end_fusion
p = 'infer_StyleStudio.py'
s = open(p).read()
if 'args.style_scale' not in s:
    s = s.replace('style_scale=1.0,',  'style_scale=args.style_scale,')
    s = s.replace('guidance_scale=5,', 'guidance_scale=args.guidance_scale,')
    s = s.replace('parser.add_argument("--neg_style_path", type=str, default=None)',
                  'parser.add_argument("--neg_style_path", type=str, default=None)\n'
                  '    parser.add_argument("--style_scale", type=float, default=1.0)\n'
                  '    parser.add_argument("--guidance_scale", type=float, default=5.0)')
    open(p,'w').write(s)
    print('Exposed style_scale & guidance_scale.')
patch_forward_end_fusion('infer_StyleStudio.py', ' ' * 8)

# layout stability co CUNG bug -> chi can forward end_fusion
patch_forward_end_fusion('infer_StyleStudio_layout_stability.py', ' ' * 12)
print('Done.')

In [ ]:
import subprocess, os
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs('sweep', exist_ok=True)
efs = [1, 5, 10, 20, 50]
for ef in efs:
    cmd = (f'HF_HUB_DISABLE_XET=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python infer_StyleStudio.py '
           f'--prompt "A red apple" --style_path "assets/style1.jpg" '
           f'--adainIP --fuSAttn --end_fusion {ef} --num_inference_steps 50')
    subprocess.run(cmd, shell=True, check=True)
    os.replace('test.jpg', f'sweep/apple_ef{ef:02d}.jpg')
    print('done end_fusion =', ef)

# ghép 1 hàng để so như figure paper
fig, ax = plt.subplots(1, len(efs)+1, figsize=(3*(len(efs)+1), 3))
ax[0].imshow(Image.open('assets/style1.jpg')); ax[0].set_title('Style'); ax[0].axis('off')
for i, ef in enumerate(efs):
    ax[i+1].imshow(Image.open(f'sweep/apple_ef{ef:02d}.jpg'))
    ax[i+1].set_title(f'end_fusion={ef}'); ax[i+1].axis('off')
plt.tight_layout(); plt.savefig('sweep/compare.jpg', dpi=110); plt.show()

## 5b. Kiểm tra Adaptive Fusion (mới — nhánh `adaptive-fusion`)

Smoke test theo `docs/superpowers/plans/2026-07-13-adaptive-end-fusion.md` (Task 6): chạy unit test trên GPU thật, rồi so sánh 1 case `--adaptive_fusion` với 1 case `--end_fusion 20` cố định (25 bước cho nhanh).

Tiêu chí PASS:
1. `pytest` xanh.
2. Case adaptive không OOM; `stop_step` nằm trong `[5, 30]`; `r_history` giảm dần sau vài bước đầu (được phép tăng nhẹ 2-3 bước đầu do baseline running-max).
3. Ảnh adaptive nhìn hợp lý, so sánh bằng mắt với fixed20.
4. `elapsed_sec` của adaptive ≤ fixed20 + 10%.

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -q

In [ ]:
import subprocess, os, json, shutil

os.makedirs('smoke', exist_ok=True)

def run_case(name, extra_args):
    cmd = (f'HF_HUB_DISABLE_XET=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python infer_StyleStudio.py '
           f'--prompt "A goat is playing on the beach" --style_path "assets/style1.jpg" '
           f'--adainIP --fuSAttn --num_inference_steps 25 '
           f'--log_json smoke/{name}.json {extra_args}')
    subprocess.run(cmd, shell=True, check=True)
    shutil.copy('test.jpg', f'smoke/{name}.jpg')
    print('done:', name)

run_case('adaptive', '--adaptive_fusion --rho 0.2 --end_fusion_max 30')
run_case('fixed20',  '--end_fusion 20')

for name in ['adaptive', 'fixed20']:
    d = json.load(open(f'smoke/{name}.json'))
    fusion = d.get('fusion') or {}
    print(f"\n[{name}] elapsed={d['elapsed_sec']}s  stop_step={fusion.get('stop_step')}")
    if fusion.get('r_history'):
        print('  r(t):', [round(r, 3) for _, r in fusion['r_history']])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(Image.open('assets/style1.jpg')); ax[0].set_title('Style'); ax[0].axis('off')
ax[1].imshow(Image.open('smoke/adaptive.jpg')); ax[1].set_title('adaptive_fusion'); ax[1].axis('off')
ax[2].imshow(Image.open('smoke/fixed20.jpg')); ax[2].set_title('end_fusion=20'); ax[2].axis('off')
plt.tight_layout(); plt.show()

## 6. Xem kết quả

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

style = Image.open('assets/style1.jpg')
result = Image.open('test.jpg')

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(style);  ax[0].set_title('Style reference'); ax[0].axis('off')
ax[1].imshow(result); ax[1].set_title('StyleStudio output'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 7. (Tùy chọn) Style-based CFG với negative style image
Đóng góp thứ 3 của paper — kiểm soát chọn lọc phần tử style, dùng ảnh negative style.

In [ ]:
!HF_HUB_DISABLE_XET=1 python infer_StyleStudio.py \
  --prompt "A red beer" \
  --style_path "assets/style2.jpg" \
  --neg_style_path "assets/neg_style2.jpg"

from PIL import Image

display(Image.open('test.jpg'))

## 8. Kiểm chứng `end_fusion` & so sánh nâng cao

Cơ chế Teacher Model: sample 0 = teacher (KHÔNG style, layout sạch), sample 1 = student (có style, là ảnh được lưu). `fuSAttn` ép **self-attention** của student bám theo teacher trong `end_fusion` bước đầu. Early steps quyết định bố cục, late steps quyết định kết cấu → `end_fusion` càng lớn student càng bám kết cấu (ảnh thật) của teacher → **style/nét cọ giảm dần**.

- **(A)** Đo pixel-diff giữa các `end_fusion` → khẳng định ảnh có KHÁC nhau (giống hệt từng pixel mới là bug).
- **(B)** Sweep **không `--adainIP`** (style cộng): bỏ AdaIN ép thống kê style mỗi bước → hiệu ứng end_fusion lên kết cấu rõ hơn.
- **(C)** `infer_StyleStudio_layout_stability.py`: nhiều style dùng **chung teacher noise** → tái hiện figure layout-stability của paper.

> (A) cần cell **sweep (mục trên)** đã chạy (tạo `sweep/`). (B), (C) tự chạy inference riêng nên khá lâu; OOM thì giảm `num_inference_steps`.

In [ ]:
# (A) Pixel-diff giữa các end_fusion — cần cell sweep phía trên đã chạy (thư mục sweep/)
import numpy as np
from PIL import Image

efs = [1, 5, 10, 20, 50]
imgs = {ef: np.asarray(Image.open(f'sweep/apple_ef{ef:02d}.jpg').convert('RGB'), dtype=np.float32)
        for ef in efs}
base = imgs[efs[0]]

print(f'{"end_fusion":>10} | {"mean|d| vs ef=1":>16} | {"% pixel lech >3":>16}')
print('-' * 50)
for ef in efs:
    d = np.abs(imgs[ef] - base)
    print(f'{ef:>10} | {d.mean():>16.3f} | {(d.max(axis=2) > 3).mean()*100:>15.2f}%')

allsame = all(np.array_equal(imgs[ef], base) for ef in efs)
print('\n' + ('[X] Moi anh GIONG HET -> end_fusion KHONG tac dung (bug).'
              if allsame else
              '[OK] Cac anh khac nhau ve so -> end_fusion dang co tac dung (khac biet co the tinh te bang mat).'))

In [ ]:
# (B) Sweep KHONG --adainIP (style cong) de thay ro end_fusion tac dong len ket cau/style
import subprocess, os
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs('sweep_noadain', exist_ok=True)
efs = [1, 5, 10, 20, 50]
for ef in efs:
    cmd = ('HF_HUB_DISABLE_XET=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python infer_StyleStudio.py '
           '--prompt "A red apple" --style_path "assets/style1.jpg" '
           f'--fuSAttn --end_fusion {ef} --num_inference_steps 50')   # KHONG co --adainIP
    subprocess.run(cmd, shell=True, check=True)
    os.replace('test.jpg', f'sweep_noadain/apple_ef{ef:02d}.jpg')
    print('done (no adain) end_fusion =', ef)

fig, ax = plt.subplots(1, len(efs) + 1, figsize=(3 * (len(efs) + 1), 3))
ax[0].imshow(Image.open('assets/style1.jpg')); ax[0].set_title('Style'); ax[0].axis('off')
for i, ef in enumerate(efs):
    ax[i + 1].imshow(Image.open(f'sweep_noadain/apple_ef{ef:02d}.jpg'))
    ax[i + 1].set_title(f'ef={ef} (no adain)'); ax[i + 1].axis('off')
plt.tight_layout(); plt.savefig('sweep_noadain/compare.jpg', dpi=110); plt.show()

In [ ]:
# (C) Layout stability: nhieu style dung CHUNG teacher noise -> bo cuc nhat quan (figure cua paper)
import subprocess, os, shutil
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs('stability_styles', exist_ok=True)
for s in ['style1.jpg', 'style2.jpg', 'style3.jpg']:
    shutil.copy(f'assets/{s}', f'stability_styles/{s}')

cmd = ('HF_HUB_DISABLE_XET=1 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python infer_StyleStudio_layout_stability.py '
       '--prompt "A red apple" --style_path "stability_styles" '
       '--adainIP --fuSAttn --end_fusion 20 --num_inference_steps 50')
subprocess.run(cmd, shell=True, check=True)   # ghi ra concat_test.jpg (noi doc cac ket qua)

result = Image.open('concat_test.jpg')
plt.figure(figsize=(4, 12))
plt.imshow(result); plt.axis('off')
plt.title('Layout stability — chung teacher noise\n(bo cuc qua tao nhat quan giua cac style khac nhau)')
plt.show()
# Meo: bo --fuSAttn o lenh tren roi chay lai de thay bo cuc "troi" khi KHONG co teacher model.

## 🛠️ Xử lý lỗi thường gặp

**`CUDA out of memory`** (T4/P100 có 16GB, chế độ Teacher Model chạy batch 2 ở 1024×1024 khá căng):
chèn thêm 2 dòng vào `infer_StyleStudio.py` ngay sau dòng `pipe.enable_vae_tiling()` rồi chạy lại — ví dụ dùng cell dưới:

```python
with open('infer_StyleStudio.py') as f: s = f.read()
s = s.replace('pipe.enable_vae_tiling()',
              'pipe.enable_vae_tiling()\n    pipe.enable_attention_slicing()')
with open('infer_StyleStudio.py','w') as f: f.write(s)
```

> Lưu ý (từ review nhánh `adaptive-fusion`): `enable_attention_slicing()` cài đặt `SlicedAttnProcessor` cho UNet, nhưng `StyleStudio_Adapter.set_ip_adapter()` chạy sau đó và **thay toàn bộ attention processor** bằng bản hijack (teacher/student, có/không adaptive) — nên slicing có thể bị mất tác dụng. Nếu OOM ngay cả với dòng patch trên, nghi ngờ đầu tiên là chỗ này, không phải do thiếu patch.

- Nếu vẫn OOM: giảm `--num_inference_steps`, hoặc chạy KHÔNG bật teacher model (bỏ `--fuSAttn`, khi đó batch = 1). Với `--adaptive_fusion`, dừng sớm cũng giảm bớt áp lực VRAM so với `--end_fusion` cố định lớn.
- **Tải model chậm/timeout**: chạy lại cell 4 (đã cache một phần), đảm bảo Internet = On.
- **Cảnh báo version torch**: bỏ qua — dùng torch có sẵn của Kaggle là đúng chủ đích.

> Lưu ý Kaggle: phiên tối đa **12 giờ**, quota **30 giờ GPU/tuần**. Đây là môi trường chạy thử/tái lập, không phải hosting 24/7.